In [4]:
import pandas as pd
import os
import glob

def get_incomplete_matrix():
	# Find all binary_numt_presence.txt files
	file_paths = glob.glob("combined_matrix.t2t_*.tsv")

	if not file_paths:
		print(f"# No 'combined_matrix.t2t_*.tsv' files found matching the pattern")
		return

	# Merge per species matrices.
	merged_df = None
	for path in file_paths:
		df = pd.read_csv(path, sep=r'\s+').fillna('Missing')
		species = path.split('_')[-1].split('.')[0]
		assembly = df.iloc[0]['NUMT_ID'].split('_')[0]
		df.columns = [f"{col}_{species}" 
				if col not in ['NUMT_ID','Bonobo', 'Chimpanzee', 'Human', 'Gorilla', 'Sorang', 'Borang'] 
				else col  
				for col in df.columns]
		if merged_df is None:
			merged_df = df
		else:
			merged_df = pd.merge(merged_df, df, how='outer')

	# Values that can be inferred form the rest of the table.
	merged_df = merged_df.fillna('?')

	# Species column.
	merged_df['species'] = merged_df['NUMT_ID'].str.split('_').str[0]
	# Extract the trailing digits after '_N' (e.g., 'mGorGor1_pri_N100' -> 100)
	merged_df['sort_index'] = merged_df['NUMT_ID'].str.extract(r'_numt(\d+)$').astype(int)
	# Sort numerically by that index
	merged_df = merged_df.sort_values(['species','sort_index']).drop(columns=['species','sort_index']).reset_index(drop=True)

	merged_df = merged_df.replace({1.0:1, 0.0:0})

	# For rows where all T2T columns are 'Missing', set the matching species to 1, and the non-matching species columns to 0.
	species_cols = ['Bonobo', 'Chimpanzee', 'Human', 'Gorilla', 'Sorang', 'Borang']
	all_missing_mask = merged_df[species_cols].eq('Missing').all(axis=1)
	numt_to_species = {
		'CHM13': 'Human',
		'mGorGor': 'Gorilla',
		'mPanPan': 'Bonobo',
		'mPanTro': 'Chimpanzee',
		'mPonPyg': 'Sorang',
		'mPonAbe': 'Borang'
	}
	for idx in merged_df[all_missing_mask].index:
		numt_id = merged_df.loc[idx, 'NUMT_ID']
		matching_species = next((sp for prefix, sp in numt_to_species.items() if numt_id.startswith(prefix)), None)
		if matching_species:
			merged_df.loc[idx, species_cols] = 0
			merged_df.loc[idx, matching_species] = 1

	# Output.
	merged_df.to_csv("all_matrix.tsv", sep='\t', index=False)
	return merged_df


matrix = get_incomplete_matrix()
matrix.head(24)


,NUMT_ID,Bonobo,Chimpanzee,Human,Gorilla,Sorang,Borang,Amani_GorGor,Carolyn_GorGor,Delphi_GorGor,...,Vicky_PonAbe,BALDY_PonAbe,Jeff_PonAbe,LIKOE_PonAbe,Catherine_PanPan,Hermien_PanPan,Hortense_PanPan,Kombote_PanPan,Natalie_PanPan,Desmond_PanPan
0,CHM13_numt1,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
1,CHM13_numt2,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
2,CHM13_numt3,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
3,CHM13_numt4,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4,CHM13_numt5,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
5,CHM13_numt6,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
6,CHM13_numt7,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
7,CHM13_numt8,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
8,CHM13_numt9,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
9,CHM13_numt10,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?


In [2]:
# Need psl mappings of CHM13 NUMT to mGorGor1 NUMT, otherwise I cannot know how to infer the HPRC NUMT presence matching other species NUMTs.

# That way, we can tell which CHM13 NUMTs in HPRC are missing in regards to the other species NUMTs.
# Limitations: Since we are only profiling T2T NUMTs across other individuals, we are not discovering new NUMTs in HPRC that could then be shared with NUMTs in other species, and this serves as a lower bound.

# Get NUMT IDs for queries and matches in T2T results

In [3]:
def get_matches_file():
	# Import file.
	matches_file = "kmerFilter_expanded_matches_similarity_results.flanks500.tab"
	df = pd.read_table(matches_file)
	return df

def needs_numt_id():
	# Import file.
	df = get_matches_file()

	# Relevant columns that need NUMT_ID info.
	df = df[['Query_Species', 'NUMT', 'Loci', 'DB_Species', 'Matches']]
	df = df[['DB_Species', 'Matches']].drop_duplicates().reset_index(drop=True)

	# Expand matches column into BED format.
	df['Strand'] = df['Matches'].str[0]
	mask = df['Matches'].str.split(':')
	df['Chr'] = df['DB_Species'] +':'+ mask.str[-2]
	df['Start'] = mask.str[-1].str.split('-').str[0].astype(int)
	df['End'] = mask.str[-1].str.split('-').str[1].astype(int)
	df['NUMT_ID'] = '?'
	df['Score'] = '.'
	df = df[['Chr', 'Start', 'End', 'NUMT_ID', 'Score', 'Strand']]

	# Sort like bedtools sort (?).
	df = df.sort_values(['Chr', 'Start', 'End'], kind='mergesort').reset_index(drop=True)
	
	# Output.
	df.to_csv("needs_NUMT_IDs.bed", header=None, sep='\t', index=False)
	return df

needs_numt_id()

,Chr,Start,End,NUMT_ID,Score,Strand
0,CHM13:NC_060925.1,1854506,1855601,?,.,+
1,CHM13:NC_060925.1,1854525,1855597,?,.,+
2,CHM13:NC_060925.1,1854560,1855597,?,.,+
3,CHM13:NC_060925.1,5076640,5077771,?,.,+
4,CHM13:NC_060925.1,5076645,5077777,?,.,+
...,...,...,...,...,...,...
32776,mSymSyn1:chrX_hap1,149335141,149336212,?,.,+
32777,mSymSyn1:chrY_hap2,7290106,7291224,?,.,+
32778,mSymSyn1:chrY_hap2,10571091,10572389,?,.,+
32779,mSymSyn1:chrY_hap2,11234445,11235578,?,.,+


In [4]:
#  Annotate with NUMT_ID info.
!bedtools intersect -a needs_NUMT_IDs.bed -b joint.dict.v1_to_v2.bed -wao > resolved.needs_NUMT_IDs.bed

In [147]:
def get_query_to_matches():
	# Import file.
	matches_file = "resolved.needs_NUMT_IDs.bed"
	df = pd.read_table(matches_file, header=None)

	# Get resolved NUMT IDs of matches.
	df = df[[ 0,1,2,5,9 ]]
	df.columns = [ 'Chr', 'Start', 'End', 'Strand', 'Matches_NUMT' ]
	df['Matches'] = df.Strand +'::'+ df.Chr.str.split(':').str[-1] +':'+ df.Start.astype(str) +'-'+ df.End.astype(str)

	# Import matches.
	matches = get_matches_file()
	# Relevant columns that need NUMT_ID info.
	matches = matches[['Query_Species', 'NUMT', 'Loci', 'DB_Species', 'Matches']]
	matches = matches.rename({'NUMT':'Query_NUMT'}, axis=1)

	# Merge NUMT_IDs for queries and matches.
	merged_df = pd.merge(matches, df, on='Matches', how='right')
	merged_df = merged_df[[ 'Query_Species', 'Query_NUMT', 'DB_Species', 'Matches_NUMT' ]]
	merged_df = merged_df.drop_duplicates()

	# Extract the trailing digits after '_N' (e.g., 'mGorGor1_pri_N100' -> 100)
	merged_df['sort_index'] = merged_df['Query_NUMT'].str.extract(r'_numt(\d+)$').astype(int)
	# Sort numerically by that index
	merged_df = merged_df.sort_values(['Query_Species','sort_index']).drop(columns=['sort_index']).reset_index(drop=True)

	# Annotate missing values.
	merged_df.loc[merged_df['Matches_NUMT']=='.', 'Matches_NUMT'] = '(not_in_v1)'

	# Remove matches to the same NUMT.
	merged_df = merged_df[merged_df['Query_NUMT']!=merged_df['Matches_NUMT']]
	# Remove matches to the same species.
	merged_df = merged_df[merged_df['Query_Species']!=merged_df['DB_Species']]

	# Drop siamang.
	merged_df = merged_df[
		~merged_df['Query_NUMT'].str.contains('mSymSyn1', na=False) &
		~merged_df['Matches_NUMT'].str.contains('mSymSyn1', na=False)
	].reset_index(drop=True)

	# Export.
	merged_df.to_csv( 'dict.query_to_matches.txt', index=None, sep='\t' )
	return merged_df

qtom = get_query_to_matches()
qtom

,Query_Species,Query_NUMT,DB_Species,Matches_NUMT
8040,mGorGor1,mGorGor1_numt1,CHM13,CHM13_numt495
8047,mGorGor1,mGorGor1_numt2,CHM13,CHM13_numt495
8048,mGorGor1,mGorGor1_numt2,CHM13,CHM13_numt496
8056,mGorGor1,mGorGor1_numt3,CHM13,CHM13_numt496
8061,mGorGor1,mGorGor1_numt4,CHM13,CHM13_numt497
...,...,...,...,...
40047,mPonPyg2,mPonPyg2_numt771,CHM13,CHM13_numt774
40050,mPonPyg2,mPonPyg2_numt772,CHM13,CHM13_numt773
40051,mPonPyg2,mPonPyg2_numt772,CHM13,CHM13_numt774
40055,mPonPyg2,mPonPyg2_numt774,CHM13,CHM13_numt771


# Use query to matches of NUMTs to infer values in new individuals

In [67]:
matrix

,NUMT_ID,Bonobo,Chimpanzee,Human,Gorilla,Sorang,Borang,Amani_GorGor,Carolyn_GorGor,Delphi_GorGor,...,Vicky_PonAbe,BALDY_PonAbe,Jeff_PonAbe,LIKOE_PonAbe,Catherine_PanPan,Hermien_PanPan,Hortense_PanPan,Kombote_PanPan,Natalie_PanPan,Desmond_PanPan
0,CHM13_numt1,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
1,CHM13_numt2,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
2,CHM13_numt3,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
3,CHM13_numt4,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4,CHM13_numt5,0,0,1,0,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4764,mPonPyg2_numt804,0,0,0,0,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4765,mPonPyg2_numt805,0,0,0,0,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4766,mPonPyg2_numt806,0,0,0,0,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4767,mPonPyg2_numt807,0,0,0,0,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?


In [198]:
def assume_zeroes_and_ones():
	df = get_incomplete_matrix()

	sample = 'hprc'
	species = 'Human'

	sample_cols = df.columns[df.columns.str.contains(sample)].to_list()
	cols_to_keep = list(df.columns[:7]) + sample_cols
	df = df[cols_to_keep]

	# For rows where all T2T columns are 'Missing', set the matching species to 1, and the non-matching species columns to 0.
	species_cols = ['Bonobo', 'Chimpanzee', 'Human', 'Gorilla', 'Sorang', 'Borang']
	all_missing_mask = df[species_cols].eq('Missing').all(axis=1)
	numt_to_species = {
		'CHM13': 'Human',
		'mGorGor': 'Gorilla',
		'mPanPan': 'Bonobo',
		'mPanTro': 'Chimpanzee',
		'mPonPyg': 'Sorang',
		'mPonAbe': 'Borang'
	}
	for idx in df[all_missing_mask].index:
		numt_id = df.loc[idx, 'NUMT_ID']
		matching_species = next((sp for prefix, sp in numt_to_species.items() if numt_id.startswith(prefix)), None)
		if matching_species:
			df.loc[idx, species_cols] = 0
			df.loc[idx, matching_species] = 1

	# For each row, if all species columns have 1, set all sample columns to 1
	species_cols = ['Bonobo', 'Chimpanzee', 'Human', 'Gorilla', 'Sorang', 'Borang']
	mask = df[species_cols].eq(1).all(axis=1)
	df.loc[mask, sample_cols] = 1

	# For each row, if Human column has 0, set all sample columns to 0
	mask = df[species] == 0
	df.loc[mask, sample_cols] = 0

	return df

df_assumed = assume_zeroes_and_ones()
df_assumed[df_assumed.isin(['?']).any(axis=1)]

,NUMT_ID,Bonobo,Chimpanzee,Human,Gorilla,Sorang,Borang,HG00438_hprc,HG00621_hprc,HG00673_hprc,...,HG03098_hprc,HG03453_hprc,HG03486_hprc,HG03492_hprc,HG03516_hprc,HG03540_hprc,HG03579_hprc,NA18906_hprc,NA20129_hprc,NA21309_hprc
800,mGorGor1_numt21,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
801,mGorGor1_numt22,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
802,mGorGor1_numt23,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
803,mGorGor1_numt24,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
804,mGorGor1_numt25,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4647,mPonPyg2_numt687,0,0,1,1,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4654,mPonPyg2_numt694,0,0,1,1,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4684,mPonPyg2_numt724,0,0,1,1,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4713,mPonPyg2_numt753,0,0,1,0,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?


In [204]:
def infer_values():
	df = assume_zeroes_and_ones()

	# Keep a subset.
	sample = 'hprc'
	species = 'Human'
	print(f'# Infer missing (?) values for {sample} samples.')
	sample_cols = df.columns[df.columns.str.contains(sample)].to_list()
	cols_to_keep = list(df.columns[:7]) + sample_cols
	df = df[cols_to_keep]

	# Only use queries that have a CHM13 value as a match.
	queries_n_matches = get_query_to_matches()
	queries_n_matches = queries_n_matches[queries_n_matches['DB_Species']=='CHM13']
	# Prepare query to matches dictionary.
	qtom = queries_n_matches.copy()
	dict_qtom = dict(zip(qtom['Query_NUMT'], qtom['Matches_NUMT']))
	# Prepare matches to query dictionary.
	mtoq = queries_n_matches.copy()
	mtoq = mtoq[mtoq['Matches_NUMT'] != '(not_in_v1)']
	dict_mtoq = dict(zip(mtoq['Matches_NUMT'], mtoq['Query_NUMT']))

	for query_numt in queries_n_matches['Query_NUMT']:
		query_index = df[df['NUMT_ID']==query_numt].index[0]

		if '?' not in df.loc[query_index, sample_cols].values:
			print(f'# There is no missing (?) values for {sample} samples at {query_numt}. Skipping...')
		try:
			matching_numt = dict_qtom[query_numt]
			matching_index = df[df['NUMT_ID']==matching_numt].index[0]
		except:
			print(f'# This numt ({matching_numt}) does not have a corresponding query.')
			break

		print(f'# Query: {query_numt}, Matching: {matching_numt}' )

		if matching_numt == '(not_in_v1)':
			print(f'# Query NUMT ({query_numt}) did not have a match corresponding to NUMT_ID_v1.')
		else:
			value_in_t2t = df[df['NUMT_ID']==matching_numt][species].to_list()[0]
			print( f'# T2T {species} at matching NUMT: ', value_in_t2t )

			# Infer missing values.
			if value_in_t2t == 0:
				print(f'# All {sample} samples should have 0 at {query_numt}.')
			elif value_in_t2t == 1:
				print(f'# The {sample} samples could have 1 or 0 at {query_numt}. Copying the row values at {matching_numt} to the query {query_numt}.')
				df.loc[query_index, sample_cols] = df.loc[matching_index, sample_cols]
				print('ok')
			else:
				print(f"# Don't know what to do since the value in {species} {matching_numt} is not 0 or 1.")
	return df


df_inferred = infer_values()
df_inferred

# Infer missing (?) values for hprc samples.
# There is no missing (?) values for hprc samples at mGorGor1_numt1. Skipping...
# Query: mGorGor1_numt1, Matching: CHM13_numt495
# T2T Human at matching NUMT:  1
# The hprc samples could have 1 or 0 at mGorGor1_numt1. Copying the row values at CHM13_numt495 to the query mGorGor1_numt1.
ok
# There is no missing (?) values for hprc samples at mGorGor1_numt2. Skipping...
# Query: mGorGor1_numt2, Matching: CHM13_numt496
# T2T Human at matching NUMT:  1
# The hprc samples could have 1 or 0 at mGorGor1_numt2. Copying the row values at CHM13_numt496 to the query mGorGor1_numt2.
ok
# There is no missing (?) values for hprc samples at mGorGor1_numt2. Skipping...
# Query: mGorGor1_numt2, Matching: CHM13_numt496
# T2T Human at matching NUMT:  1
# The hprc samples could have 1 or 0 at mGorGor1_numt2. Copying the row values at CHM13_numt496 to the query mGorGor1_numt2.
ok
# There is no missing (?) values for hprc samples at mGorGor1_numt3. Skipping...
#

,NUMT_ID,Bonobo,Chimpanzee,Human,Gorilla,Sorang,Borang,HG00438_hprc,HG00621_hprc,HG00673_hprc,...,HG03098_hprc,HG03453_hprc,HG03486_hprc,HG03492_hprc,HG03516_hprc,HG03540_hprc,HG03579_hprc,NA18906_hprc,NA20129_hprc,NA21309_hprc
0,CHM13_numt1,0,0,1,0,0,0,1,1,1,...,1,1,1,1,1,1,1,1,1,1
1,CHM13_numt2,0,0,1,0,0,0,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2,CHM13_numt3,0,0,1,0,0,0,1,1,1,...,1,1,1,1,1,1,1,1,1,1
3,CHM13_numt4,0,0,1,0,0,0,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4,CHM13_numt5,0,0,1,0,0,0,1,1,1,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4764,mPonPyg2_numt804,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4765,mPonPyg2_numt805,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4766,mPonPyg2_numt806,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4767,mPonPyg2_numt807,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [200]:
df_inferred[df_inferred.isin(['?']).any(axis=1)]

,NUMT_ID,Bonobo,Chimpanzee,Human,Gorilla,Sorang,Borang,HG00438_hprc,HG00621_hprc,HG00673_hprc,...,HG03098_hprc,HG03453_hprc,HG03486_hprc,HG03492_hprc,HG03516_hprc,HG03540_hprc,HG03579_hprc,NA18906_hprc,NA20129_hprc,NA21309_hprc
800,mGorGor1_numt21,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
801,mGorGor1_numt22,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
802,mGorGor1_numt23,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
803,mGorGor1_numt24,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
804,mGorGor1_numt25,1,1,1,1,0,0,?,?,?,...,?,?,?,?,?,?,?,?,?,?
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4647,mPonPyg2_numt687,0,0,1,1,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4654,mPonPyg2_numt694,0,0,1,1,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4684,mPonPyg2_numt724,0,0,1,1,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4713,mPonPyg2_numt753,0,0,1,0,1,1,?,?,?,...,?,?,?,?,?,?,?,?,?,?
